# 🤖 Conversational AI Chatbot — Open-Source Dialogue Assistant with Multi-Turn Memory

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mitadrudeb/building-genai-powered-applications/blob/main/02-conversational-ai-chatbot/Conversational_AI_Chatbot.ipynb)

**Author:** [Mitadru Deb](https://github.com/mitadrudeb)  
**Repository:** [building-genai-powered-applications](https://github.com/mitadrudeb/building-genai-powered-applications)  
**Focus:** Beginner AI / LLM Engineering & Conversational Dialogue Systems  
**Model:** [facebook/blenderbot-400M-distill](https://huggingface.co/facebook/blenderbot-400M-distill) (~400M parameters)  
**License:** MIT

---

## 📌 Project Overview

Welcome to **Project 02** of the **Building GenAI-Powered Applications** series! In this project, we build a complete, production-ready **Conversational AI Chatbot** using open-source tools.

Large Language Models and conversational agents are fundamentally **stateless** — they do not remember what was said in previous turns unless the dialogue history is explicitly managed and fed back into the prompt. In this notebook, you will learn:

1. **Seq2Seq Dialogue Architecture:** How encoder-decoder models like Facebook BlenderBot generate empathetic and coherent multi-turn responses.
2. **Conversational Memory Management:** How to implement a robust **sliding-window memory buffer** to maintain conversation continuity without exceeding model context limits.
3. **Decoding Strategies & Generation Hyperparameters:** How **Temperature**, **Top-P (Nucleus Sampling)**, **Repetition Penalty**, and **N-gram Blocking** shape the personality, variety, and reliability of the output.
4. **Interactive Web Interface:** How to wrap the conversational engine into a modern **Gradio** web application with tunable controls and one-click cloud deployment via Google Colab.
5. **Frontier Model Extension:** How to map open-source conversational memory to industry-standard commercial APIs (OpenAI Chat Completions).

> 💡 **Hardware Friendly:** This project uses an optimized ~400M parameter model that runs smoothly on standard consumer **CPUs** (<1s per response) and automatically scales to **NVIDIA GPUs** when available. Zero external API keys required!

---
## 🧠 1. Theoretical Foundations

### 🔄 Seq2Seq (Encoder-Decoder) vs. Decoder-Only Models
- **Decoder-Only Models (e.g., GPT-4, Llama 3, Qwen):** Predict the next token autoregressively from left to right given the entire concatenated context.
- **Encoder-Decoder Models (e.g., BlenderBot, T5, BART):** The **encoder** processes the entire multi-turn conversation into bidirectional hidden representations, while the **decoder** generates the bot's response using cross-attention over those encoded states. BlenderBot was specifically pre-trained on diverse conversational corpora (PersonaChat, EmpatheticDialogues, ConvAI2, Wizard of Wikipedia) to excel at engaging human chit-chat.

### 🪟 The Sliding-Window Memory Buffer
Because transformer models have a fixed maximum sequence length (512 tokens for BlenderBot-400M), concatenating an infinite conversation history will quickly trigger memory overflows or model degradation.

A **Sliding Window** retains only the most recent $K$ dialogue turns (e.g., the last 6 user-bot exchanges). When turn $K+1$ arrives, the oldest exchange drops out of active memory:

```
[Turn 1: User / Bot] -> [Turn 2: User / Bot] -> [Turn 3: User / Bot] -> [Current User Prompt]
      └───────────────────── Active Context Window ──────────────────────┘
```

### ⚙️ Text Generation Parameters Explained
| Parameter | What it Does | Typical Range | Recommended |
| :--- | :--- | :---: | :---: |
| `temperature` | Controls randomness. Lower = deterministic & focused; Higher = creative & varied. | `0.1 – 1.5` | `0.6 – 0.8` |
| `top_p` | Nucleus sampling: only samples from candidate tokens whose cumulative probability is $\le p$. | `0.1 – 1.0` | `0.85 – 0.9` |
| `repetition_penalty` | Discount factor applied to previously generated tokens to prevent repetitive loops. | `1.0 – 2.0` | `1.2 – 1.4` |
| `no_repeat_ngram_size` | Strictly forbids the model from repeating any sequence of $n$ consecutive words. | `2 – 4` | `3` |
| `max_new_tokens` | Upper bound on the length of the generated reply. | `20 – 150` | `60` |

---
## 📦 2. Environment Setup & Dependencies

Install the required open-source libraries: `transformers`, `torch`, `gradio`, and `sentencepiece`.

In [ ]:
# Install required dependencies (uncomment if running in a fresh environment/Colab)
# !pip install -q transformers torch gradio sentencepiece accelerate openai

import torch
import transformers
import gradio as gr

print(f"🔥 PyTorch Version:      {torch.__version__}")
print(f"🤗 Transformers Version: {transformers.__version__}")
print(f"🎨 Gradio Version:       {gr.__version__}")
print(f"💻 CUDA Available:       {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"⚡ GPU Device:           {torch.cuda.get_device_name(0)}")
else:
    print("ℹ️ Running on CPU (Optimized for lightweight inference)")

---
## 🚀 3. Loading Model & Tokenizer

We load `facebook/blenderbot-400M-distill` directly from the Hugging Face Hub using `AutoTokenizer` and `AutoModelForSeq2SeqLM`. The model weights are cached locally so subsequent runs load instantly without re-downloading.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import time

MODEL_NAME = "facebook/blenderbot-400M-distill"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {MODEL_NAME} onto {DEVICE.upper()}...")
start_time = time.time()

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)

load_duration = time.time() - start_time
print(f"✅ Model & Tokenizer loaded in {load_duration:.2f} seconds!")
print(f"📊 Total Parameters: ~400 Million (Distilled for high performance)")

---
## 🛠️ 4. Conversational Generation Engine with Memory Management

### Robust Message Parsing
In modern web frameworks like Gradio 5/6, messages can arrive as plain strings, dictionaries (`{'text': ...}`), or lists of content blocks (`['...']`). We include a simple helper function `extract_text` to safely extract plain text strings from any message format, ensuring the application never crashes during multi-turn conversations.

In [ ]:
def extract_text(item) -> str:
    """
    Safely extract plain text from string, list, or dict.
    Ensures full compatibility across Gradio versions and CLI inputs.
    """
    if not item:
        return ""
    if isinstance(item, str):
        return item.strip()
    if isinstance(item, dict):
        if "text" in item:
            return extract_text(item["text"])
        if "content" in item:
            return extract_text(item["content"])
    if isinstance(item, (list, tuple)):
        parts = [extract_text(sub) for sub in item]
        return " ".join(p for p in parts if p).strip()
    return str(item).strip()


def generate_bot_response(
    message,
    history,
    temperature: float = 0.7,
    top_p: float = 0.85,
    repetition_penalty: float = 1.3,
    max_tokens: int = 60,
    history_turns: int = 6,
) -> str:
    """
    Generate a context-aware conversational response using multi-turn memory.
    
    Args:
        message: Latest user input (string or dict).
        history: List of previous conversation turns.
        temperature: Randomness in sampling (0.1 = deterministic, 1.5 = highly creative).
        top_p: Nucleus sampling threshold.
        repetition_penalty: Penalty factor for repeating previously generated tokens.
        max_tokens: Maximum tokens to generate.
        history_turns: Number of preceding turns retained in the sliding window.
    """
    user_text = extract_text(message)
    if not user_text:
        return "Please say something so we can chat!"

    # 1. Parse history into individual conversational utterances
    utterances = []
    if history:
        for turn in history:
            if isinstance(turn, dict):
                # Gradio messages format: {'role': 'user'/'assistant', 'content': ...}
                text = extract_text(turn.get("content"))
                if text:
                    utterances.append(text)
            elif isinstance(turn, (list, tuple)) and len(turn) == 2:
                # Gradio legacy tuples format: (user_msg, bot_msg)
                user_msg = extract_text(turn[0])
                bot_msg = extract_text(turn[1])
                if user_msg:
                    utterances.append(user_msg)
                if bot_msg:
                    utterances.append(bot_msg)

    # 2. Sliding window memory: keep only recent dialogue turns (N turns = 2N utterances)
    max_history_count = int(history_turns) * 2
    recent_utterances = utterances[-max_history_count:]
    recent_utterances.append(user_text)

    # 3. BlenderBot native delimiter: double space ("  ") between utterances
    prompt = "  ".join(recent_utterances)

    # 4. Tokenize with safe truncation
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to(DEVICE)

    # 5. Generate conversational response with sampling controls
    outputs = model.generate(
        **inputs,
        max_new_tokens=int(max_tokens),
        no_repeat_ngram_size=3,
        repetition_penalty=float(repetition_penalty),
        do_sample=True,
        temperature=float(temperature),
        top_p=float(top_p),
    )

    # 6. Decode output tokens back into natural language
    response = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    return response

print("✅ Generation engine initialized and ready!")

---
## 🧪 5. Testing Multi-Turn Dialogue Memory (Step-by-Step)

Let's test whether the bot can resolve pronouns (e.g. *"it"*, *"that place"*) across multiple turns. Notice how the second turn references the topic established in the first turn!

In [ ]:
test_history = []

# --- Turn 1 ---
turn1_user = "Hello! What is your favorite book?"
turn1_bot = generate_bot_response(turn1_user, test_history)
print(f"👤 User: {turn1_user}")
print(f"🤖 Bot:  {turn1_bot}\n")

# Save to history
test_history.append({"role": "user", "content": turn1_user})
test_history.append({"role": "assistant", "content": turn1_bot})

# --- Turn 2 (Pronoun reference: 'it') ---
turn2_user = "Why do you like it so much?"
turn2_bot = generate_bot_response(turn2_user, test_history)
print(f"👤 User: {turn2_user}")
print(f"🤖 Bot:  {turn2_bot}\n")

# Save to history
test_history.append({"role": "user", "content": turn2_user})
test_history.append({"role": "assistant", "content": turn2_bot})

# --- Turn 3 (Topic continuation) ---
turn3_user = "Can you recommend another one just like it?"
turn3_bot = generate_bot_response(turn3_user, test_history)
print(f"👤 User: {turn3_user}")
print(f"🤖 Bot:  {turn3_bot}")

---
## 🎨 6. Building the Interactive Gradio Web Application

We now package our conversational engine into an interactive Gradio web application featuring:
- **Real-time generation sliders** (Temperature, Top-P, Repetition Penalty, Max Tokens).
- **Memory Window control** to dynamically adjust history length.
- **Curated prompt suggestions** for quick testing.
- **Public share link** via `share=True` for instant access on Google Colab.

In [ ]:
description_markdown = """
### 💬 Open-Source Conversational Dialogue Agent
Powered by **Facebook BlenderBot-400M-distill** — a lightweight (~400M parameter) Seq2Seq model 
trained specifically on multi-turn empathetic and engaging dialogue. 
Runs 100% locally on CPU/GPU without external API keys!
"""

sample_prompts = [
    ["Hello! What is your favorite book?"],
    ["If you could visit any planet, where would you go?"],
    ["Can you explain how neural networks learn in simple terms?"],
    ["What are some fun hobbies to pick up on the weekend?"],
    ["Tell me an interesting fact about space exploration."],
]

demo = gr.ChatInterface(
    fn=generate_bot_response,
    title="🤖 Conversational AI Chatbot",
    description=description_markdown,
    examples=sample_prompts,
    additional_inputs=[
        gr.Slider(
            minimum=0.1,
            maximum=1.5,
            value=0.7,
            step=0.05,
            label="🌡️ Temperature (Creativity)",
            info="Higher = creative & diverse; Lower = focused & deterministic.",
        ),
        gr.Slider(
            minimum=0.1,
            maximum=1.0,
            value=0.85,
            step=0.05,
            label="🎯 Top-P (Nucleus Sampling)",
            info="Cumulative probability threshold for candidate tokens.",
        ),
        gr.Slider(
            minimum=1.0,
            maximum=2.0,
            value=1.3,
            step=0.05,
            label="🔄 Repetition Penalty",
            info="Penalizes repetitive words or token cycles.",
        ),
        gr.Slider(
            minimum=20,
            maximum=150,
            value=60,
            step=10,
            label="📏 Max New Tokens",
            info="Maximum number of tokens to generate per reply.",
        ),
        gr.Slider(
            minimum=2,
            maximum=12,
            value=6,
            step=2,
            label="🧠 Context Memory Window (Turns)",
            info="Number of preceding dialogue turns retained in context.",
        ),
    ],
    additional_inputs_accordion=gr.Accordion(
        "⚙️ Generation & Memory Settings", open=False
    ),
)

# Launch the interactive app inline
# Set share=True if running in Google Colab to get a public link
demo.launch(share=False)

---
## 🏆 7. Key Takeaways & Beginner Challenges

### Summary of What You Learned
1. **Stateless vs. Stateful Systems:** LLMs possess no memory between API calls; statefulness is an engineering abstraction built by passing history into the prompt.
2. **Sliding-Window Efficiency:** Truncating old turns keeps computation predictable and fits within the model's sequence limit.
3. **Tokenizer Alignment Matters:** Matching the model's training prompt convention (such as BlenderBot's double-space separator) is critical to maintaining coherence.
4. **Decoding Control:** Adjusting temperature and repetition penalties drastically changes the dialogue quality.

### 🎯 Try These Hands-On Experiments!
- **Experiment 1:** Set `temperature=0.1` and `temperature=1.4`. How does the bot's tone and vocabulary change?
- **Experiment 2:** Set `repetition_penalty=1.0`. Can you cause the bot to get stuck in an repetitive loop?
- **Experiment 3:** Change `history_turns` from 6 down to 1. Test asking a pronoun-dependent question like *"What was the first thing I asked you?"* and observe where memory drops off.

---
## 🌐 8. Bonus: Frontier Model Integration (OpenAI API)

Anyone who has their own **OpenAI API key** can test this section with frontier models like **GPT-4o-mini** or **GPT-4o** to compare commercial cloud APIs with local open-source models.

### How to Test This Section:
1. Install the OpenAI library: `pip install openai`
2. Set your API key in your environment (`export OPENAI_API_KEY="your_key_here"` or provide it in the code cell below).
3. Uncomment the code below to run a test conversation.

In [ ]:
# =========================================================================
# 🌐 FRONTIER MODEL INTEGRATION (OPENAI CHAT COMPLETIONS API)
# =========================================================================
# Anyone with their own OpenAI API key can uncomment and run this cell.
#
# import os
# from openai import OpenAI
#
# # Fetch key from environment or replace with your own key for testing:
# api_key = os.getenv("OPENAI_API_KEY") or "YOUR_OPENAI_API_KEY_HERE"
#
# if not api_key or api_key == "YOUR_OPENAI_API_KEY_HERE":
#     print("⚠️ Please provide a valid OPENAI_API_KEY to test frontier models.")
# else:
#     client = OpenAI(api_key=api_key)
#
#     def chat_with_openai(user_message, history, model="gpt-4o-mini"):
#         """
#         Generate a multi-turn chat response using OpenAI Chat Completions.
#         Maps dialogue history directly into OpenAI's standard messages schema.
#         """
#         messages = [
#             {"role": "system", "content": "You are a helpful, empathetic, and concise AI assistant."}
#         ]
#         # Add sliding window history (last 6 turns = 12 messages)
#         for turn in history[-12:]:
#             role = turn.get("role", "user")
#             content = extract_text(turn.get("content"))
#             if content:
#                 messages.append({"role": role, "content": content})
#
#         messages.append({"role": "user", "content": user_message})
#
#         response = client.chat.completions.create(
#             model=model,
#             messages=messages,
#             temperature=0.7,
#             max_tokens=150,
#         )
#         return response.choices[0].message.content.strip()
#
#     # Quick test invocation:
#     sample_history = [
#         {"role": "user", "content": "What is machine learning in simple terms?"},
#         {"role": "assistant", "content": "Machine learning is teaching computers to learn patterns from data rather than writing explicit rules for every task."}
#     ]
#     reply = chat_with_openai("Can you give me a real-world example of that?", sample_history)
#     print("🤖 OpenAI Response:", reply)